# Jak Wojtek uczy się chodzić

Wojtek to czworonożny robot, który zaczął jako **4BarBot** na Politechnice Wrocławskiej. Chodu nie ma zaprogramowanego: uczy się go metodą uczenia ze wzmocnieniem (RL) w symulacji, a wytrenowana sieć trafia potem na fizycznego robota. Ten notebook prowadzi przez tę drogę krok po kroku. Każdy krok kończy się widokiem z MuJoCo.

Plan:

1. Wojtek stoi w MuJoCo.
2. Polityka na początku treningu.
3. Nagroda: co do niej wkładamy i jak steruje treningiem.
4. Trening na GPU.
5. Wytrenowana polityka w MuJoCo.
6. Zadania: własne eksperymenty z nagrodą i komendami.
7. Zadanie główne: Wojtek chodzi tam, gdzie każesz (panel sterowania).
8. Finał: Twoja polityka kontra ta, która jeździ na robocie.

## Robot

- 4 nogi, w każdej 3 silniki: odwodzenie biodra, biodro, kolano. Razem 12 silników sterowanych **pozycyjnie**: polityka podaje kąt docelowy, regulator PD w napędzie (na robocie MD80, w MuJoCo ten sam model serwa) zamienia go na moment.
- Nogi są czworobokami przegubowymi: poniżej silników łańcuch kinematyczny się zamyka. Za osobliwością kolana (ok. 3,2 rad) mechanizm może się przeskoczyć, dlatego cel kolana jest zawsze ograniczony.
- Masa 14 kg, wysokość stania ok. 0,125 m.
- Fizyka liczy się 250 razy na sekundę (krok 4 ms), polityka działa 50 razy na sekundę: jedna decyzja = 5 kroków fizyki.

## Skąd jest model

Wszystko leży w pakiecie ROS `ros/src/wojtek_description/`:

- `meshes/*.stl` — geometria ogniw robota z CAD-u;
- `urdf/*.urdf.xacro` — opis robota dla ROS-a (ten, którego używa prawdziwy robot i RViz);
- `mujoco/wojtek.xml` — ten sam robot zapisany w formacie MuJoCo (MJCF): ogniwa, przeguby, domknięcia czworoboków, siatki z `meshes/`, silniki i czujniki. Jest źródłem dla treningu;
- `mujoco/wojtek_mjx.xml` + `scene_mjx.xml` — wersja treningowa, **generowana** poleceniem `./training/run.sh build`. Skrypt bierze `wojtek.xml` i nanosi zmiany potrzebne w treningu: siatki przestają kolidować (stopy dostają kule, korpus prostopadłościan), korpus dostaje jawną masę, 12 silników momentowych staje się serwami PD, krok fizyki ustawiony na 4 ms. Tych plików nie edytuje się ręcznie; `scene_mjx.xml` dokłada podłogę, światło i kamerę śledzącą.

Notebook ładuje właśnie `scene_mjx.xml`, czyli dokładnie to, na czym trenuje polityka.

Środowisko Colab: **GPU** (Runtime → Change runtime type). Uruchamiaj komórki po kolei. Sesja Colab wygasa po ok. 90 min bezczynności: trening z kroku 7 zacznij, gdy masz czas go dopilnować.

## Krok 0 — Instalacja

Klonuje repozytorium i instaluje `training/` (JAX, MuJoCo MJX, MJWarp, Brax). Jeśli następna komórka nie zaimportuje bibliotek, zrób Runtime → Restart session i uruchom od początku.

In [ ]:
import os, subprocess, sys
from pathlib import Path

if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")   # renderowanie bez ekranu; przed `import mujoco`

REPO_BRANCH = "Add-notebook-with-the-guidance-how-to-train-Wojtek"   # po scaleniu: "main"
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "training" / "run.sh").exists()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "w01-tek"
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, "https://github.com/machinekind/w01-tek.git", str(REPO_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "-q", "--ff-only"])   # ponowne uruchomienie: dociągnij zmiany
TRAINING = REPO_ROOT / "training"

try:
    import wojtek_rl, fast_simplification, trimesh  # noqa: F401
except ImportError:
    import tomllib
    lock = tomllib.loads((TRAINING / "uv.lock").read_text())
    mujoco_pin = next(p["version"] for p in lock["package"] if p["name"] == "mujoco")   # ta sama wersja co w locku
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(TRAINING), f"mujoco=={mujoco_pin}",
                          "trimesh", "fast-simplification"],   # dwa ostatnie: uproszczone siatki do renderowania
                         capture_output=True, text=True)
    if res.returncode:
        print(res.stdout[-1500:], res.stderr[-3000:])
        raise SystemExit("pip install nie powiódł się (patrz wyżej)")
    # Colab ma preinstalowany nowszy plugin JAX dla CUDA 13; obok jax 0.9.2 tylko generuje błędy.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax-cuda13-plugin", "jax-cuda13-pjrt"], capture_output=True)
    print("zainstalowano; jeśli następna komórka nie działa, zrestartuj sesję i uruchom od początku")
print(REPO_ROOT, "| python", sys.version.split()[0])

In [ ]:
import shutil
import mediapy as media
import mujoco
import numpy as np

if shutil.which("ffmpeg") is None:          # mediapy potrzebuje binarki ffmpeg
    import imageio_ffmpeg
    media.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

for p in (TRAINING, REPO_ROOT / "learning"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from wojtek_rl import paths
import wojtek_kurs as kurs

# Colab nie ma OpenGL od NVIDII: MuJoCo renderuje programowo, a pełne siatki robota
# (600 tys. trójkątów) kosztują 0,8 s na klatkę. Rysujemy więc kopię modelu z uproszczonymi
# siatkami; fizyka liczy się na oryginale.
widok = kurs.Widok(cache=TRAINING / "videos" / "guide" / "lowpoly")
frame = widok.klatka

def show(frames, fps=25):
    media.show_video(np.asarray(frames), fps=fps, codec="h264")

print("mujoco", mujoco.__version__, "| model:", paths.SCENE_XML.relative_to(REPO_ROOT),
      f"| siatki do rysowania: {int(widok.rmodel.mesh_facenum.sum()):,} trójkątów")

## Krok 1 — Wojtek stoi w MuJoCo

Robot startuje z zapisanej pozy `home` i trzyma jej kąty w serwach. Nic więcej: tak wygląda „zerowa akcja” polityki.

In [ ]:
model = mujoco.MjModel.from_xml_path(str(paths.SCENE_XML))
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
data.ctrl[:] = model.key("home").ctrl

frames = []
for k in range(int(4.0 / model.opt.timestep)):       # 4 s
    mujoco.mj_step(model, data)
    if k % 10 == 0:                                   # 25 klatek/s
        frames.append(frame(data))

print(f"serwa: {model.nu}, masa {sum(model.body_mass):.1f} kg, wysokość bazy {data.qpos[2]:.3f} m")
show(frames)

## Krok 2 — Polityka na początku treningu

Polityka to sieć neuronowa: na wejściu obserwacja, na wyjściu 12 przesunięć kątów względem pozy `home`.

- **Obserwacja** (40 liczb): kąty i prędkości 12 przegubów, poprzednia akcja i **komenda** `[vx, vy, wz, wysokość]`, czyli wektor, w którym robot ma iść. Tylko to, co robot naprawdę mierzy: bez IMU, bez pozycji w świecie.
- **Akcja** (12 liczb w zakresie −1..1): cel = `home + akcja · skala`, obcięty do limitów. Skala to 0,25 rad dla odwodzenia i 0,5 rad dla biodra i kolana.
- **Sieć**: warstwy 512-256-128. W treningu do jej wyjścia dokłada się losowy szum, żeby próbowała różnych ruchów.

Tak wygląda start PPO: losowe wagi, szum, komenda 0,5 m/s do przodu. Sieć jeszcze nie wie, co komenda znaczy.

In [ ]:
KOMENDA = np.array([0.5, 0.0, 0.0, 0.125], np.float32)   # vx, vy, wz, wysokość stania
SEED = 0

model = kurs.model_z_serwem({"kp": 40.0, "kd": 1.6, "max_torque": 9.0})   # serwo jak w treningu
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
HOME = model.key("home").ctrl.copy()
qadr = np.array([model.jnt_qposadr[j] for j in model.actuator_trnid[:, 0]])
vadr = np.array([model.jnt_dofadr[j] for j in model.actuator_trnid[:, 0]])
SCALE = np.tile([0.25, 0.5, 0.5], 4)                      # zakres akcji na przegub, rad
LOW, HIGH = model.actuator_ctrlrange.T.copy()
LOW[0::3], HIGH[0::3] = -0.44, 0.44                        # limit odwodzenia i kolana jak w treningu
HIGH[2::3] = np.minimum(HIGH[2::3], 3.15)

# Aktor jak w PPO na starcie: losowe wagi, wyjście = środek i rozrzut rozkładu akcji.
rng = np.random.default_rng(SEED)
sizes = [12 + 12 + 12 + 4, 512, 256, 128, 2 * 12]
W = [rng.uniform(-1, 1, (a, b)) * np.sqrt(3.0 / a) for a, b in zip(sizes[:-1], sizes[1:])]
def aktor(obs):
    x = obs
    for w in W[:-1]:
        x = x @ w; x = x / (1 + np.exp(-x))               # SiLU
    out = x @ W[-1]
    std = np.log1p(np.exp(out[12:])) + 1e-3               # softplus, jak w Brax
    return np.tanh(out[:12] + std * rng.standard_normal(12))

frames, last_act = [], np.zeros(12, np.float32)
x0 = data.qpos[0]
for i in range(200):                                       # 4 s przy 50 Hz
    obs = np.concatenate([data.qpos[qadr] - HOME, data.qvel[vadr], last_act, KOMENDA])
    last_act = aktor(obs).astype(np.float32)
    data.ctrl[:] = np.clip(HOME + last_act * SCALE, LOW, HIGH)
    for _ in range(5):
        mujoco.mj_step(model, data)
    if i % 2 == 0:
        frames.append(frame(data))
print(f"po 4 s: przebyte {data.qpos[0] - x0:+.2f} m w kierunku komendy (cel: +2.0 m), wysokość bazy {data.qpos[2]:.3f} m")
show(frames)

## Krok 3 — Nagroda

Trening nie mówi sieci, *jak* chodzić. Mówi tylko, ile punktów dostaje za każdy krok 20 ms, a PPO zmienia wagi tak, żeby suma punktów w epizodzie rosła. Nagroda to suma składników: `dt · Σ waga · składnik`. Co można do niej włożyć:

- **Zadanie**: `tracking_lin_vel`, `tracking_ang_vel` — jak blisko prędkość robota jest komendy (kernel `exp(-błąd²/σ)`: 1 przy trafieniu, 0 daleko). To jedyne miejsce, gdzie komenda w ogóle ma znaczenie.
- **Postawa**: `orientation` (kara za przechył; aktor nie ma IMU, więc to jego jedyny „zmysł” pionu), `pose` (odchylenie od pozy stania), `stand_still` i `stand_feet_down` (ruch i uniesione stopy przy komendzie „stój”).
- **Chód**: `feet_air_time` i `high_step` (nagroda za odrywanie i unoszenie stóp), `feet_slip` (kara za poślizg stopy na ziemi).
- **Wysiłek i gładkość**: `torques`, `torque_rate`, `torque_limit` (moment, jego skoki, dobijanie do limitu), `action_rate` (skoki celów między krokami).
- **Koniec**: `termination` — kara za upadek, który kończy epizod.

Jak to wpływa na trening, w skrócie:

- Wagi to kompromis. Sam tracking daje robota, który drży i grzeje silniki; sama gładkość daje robota, który stoi. Każdy składnik ma cenę w innych.
- Sieć znajdzie luki. Za dużą karę za lądowanie omija, sunąc stopami zamiast stawiać kroki; kary za każdy krok potrafi „uniknąć” przewracając się wcześnie, bo krótszy epizod to mniej kar. Dlatego obok nagrody patrzy się na długość epizodu.
- Czego nie ma w nagrodzie, tego nie będzie w chodzie: obroty w miejscu czy chód do tyłu pojawiają się dopiero, gdy komendy je zadają, a nagroda za nie płaci.

Poniżej wagi z przepisu `locomotion_stiff_v1`: opublikowana polityka Wojtka trenowana tym przepisem 2 mld kroków. Zero oznacza składnik wyłączony.

In [ ]:
from hydra import compose, initialize_config_dir

with initialize_config_dir(config_dir=str(TRAINING / "wojtek_rl" / "conf"), version_base=None):
    hcfg = compose(config_name="config", overrides=[f"+experiment={kurs.PRESET}"])

GRUPY = {"zadanie": ["tracking_lin_vel", "tracking_ang_vel", "height_tracking"],
         "postawa": ["orientation", "pose", "stand_still", "stand_feet_down", "lin_vel_z", "ang_vel_xy"],
         "chód": ["feet_air_time", "high_step", "feet_slip", "feet_apex", "feet_landing"],
         "wysiłek": ["torques", "torque_rate", "torque_limit", "action_rate", "action_accel", "energy"],
         "koniec": ["termination"]}
scales = hcfg.task.env.reward.scales
for grupa, nazwy in GRUPY.items():
    print(f"{grupa:9s}", "  ".join(f"{n}={scales[n]:g}" for n in nazwy if n in scales))
c = hcfg.task.env.command
print(f"komendy   vx {list(c.vx)} m/s  vy {list(c.vy)} m/s  wz {list(c.wz)} rad/s  stój z p={c.zero_prob}")

## Krok 4 — Trening (GPU)

Ta sama sieć co w kroku 2, ale 2048 robotów naraz na GPU, każdy 20 s epizodu, a po każdej porcji kroków PPO poprawia wagi w stronę większej nagrody. Co jakiś czas trener ocenia politykę: `reward` to suma nagrody z epizodu, `ep_len` to jego długość w krokach (1000 = nie upadł).

Wszystko, co zmienia trening, podaje się jako nadpisania konfiguracji (Hydra). Najczęstsze:

| co | nadpisanie |
|---|---|
| waga składnika nagrody | `++task.env.reward.scales.action_rate=-0.5` |
| zakres komend | `'++task.env.command.vx=[0.3,0.6]'`, `'++task.env.command.wz=[0,0]'`, `++task.env.command.zero_prob=0` |
| pchnięcia w treningu | `++task.env.push.vel=0.3` |
| serwo | `++task.env.pd_kp=60 ++task.env.pd_kd=1.96` (robot musi dostać to samo) |
| inny start | `seed=1` |

Budżet kroków a efekt na T4 (ok. 50 tys. kroków/s):

| kroki | czas | co zwykle widać |
|---|---:|---|
| 10 mln | 4 min | robot przestaje upadać i stoi; na komendę reaguje słabo |
| 50 mln | 17 min | pierwsze kroki w stronę komendy |
| 100–200 mln | 35–70 min | chodzi w zadanym kierunku; jakość rośnie dalej |
| 2 mld | godziny na H100 | opublikowana polityka |

Próbka: 10 mln kroków. Jeśli MJWarp nie obsłuży karty, dodaj `dodatkowe=("+task.env.sim.backend=jax",)`.

In [ ]:
RUN = "guide_10m"
run_dir = kurs.trenuj(RUN, kroki=10_000_000, envs=2048)

## Krok 5 — Wytrenowana polityka w MuJoCo

Eksport zamienia checkpoint na `policy.npz` (wagi) i `policy_meta.json` (kontrakt: układ obserwacji, skala akcji, limity, serwo). Dokładnie ten plik dostaje robot. `kurs.przebieg` to pętla z kroku 2 spakowana do funkcji: odczyt przegubów → `polityka.step` → cele do serw → 5 kroków fizyki. Ta sama komenda, wagi po treningu.

In [ ]:
from wojtek_rl.np_policy import load_policy_runtime

polityka = load_policy_runtime(kurs.eksportuj(RUN))
p = kurs.przebieg(polityka, komenda=(0.5, 0.0, 0.0), sekundy=4.0, widok=widok)
print(kurs.tabela({RUN: p}))
show(p["klatki"])

## Krok 6 — Zadania

Każde zadanie to jeden trening z inną konfiguracją i ten sam przebieg co wyżej. Nadaj każdemu własną nazwę `RUN`, bo trening o istniejącej nazwie nie startuje ponownie. 10 mln kroków starcza, żeby zobaczyć różnicę względem `guide_10m`; porównuj tabelą i wideo, nie samą nagrodą.

1. **Inny start.** `seed=1` z tą samą konfiguracją. Jak bardzo wynik zależy od losowości?
2. **Bez nagrody za kroki.** `feet_air_time=0` i `high_step=0`. Czy robot w ogóle odrywa stopy, czy sunie?
3. **Gładkość.** `action_rate=-1.0` (4× większa kara). Co się dzieje z drganiami i z prędkością?
4. **Tylko do przodu.** Zawęź komendy do `vx` w 0,3–0,6 m/s, `vy` i `wz` do zera, `zero_prob=0`. Sieć nie marnuje kroków na obroty i stanie: uczy się szybciej iść, ale tylko tego.

Szablon poniżej; wpisz nazwę i nadpisania.

In [ ]:
RUN = "zadanie_2"
DODATKOWE = ("++task.env.reward.scales.feet_air_time=0", "++task.env.reward.scales.high_step=0")
kurs.trenuj(RUN, kroki=10_000_000, envs=2048, dodatkowe=DODATKOWE)

polityka = load_policy_runtime(kurs.eksportuj(RUN))
p = kurs.przebieg(polityka, komenda=(0.5, 0.0, 0.0), sekundy=4.0, widok=widok)
print(kurs.tabela({RUN: p}))
show(p["klatki"])

## Krok 7 — Zadanie główne: Wojtek chodzi tam, gdzie każesz

Wytrenuj politykę, która wykonuje dowolną komendę z zakresu przepisu: przód i tył, w bok, obrót. To pełne rozłożenie komend z kroku 3, więc potrzebuje więcej kroków niż próbka: zacznij od 100 mln (ok. 35 min na T4) i nie zamykaj karty. Panel niżej pozwala zadać wektor `vx, vy, wz` i obejrzeć, jak polityka go wykonuje; tabela mówi, jak daleko od komendy jest faktyczny ruch.

Warunek zaliczenia: przy `vx=0,5` robot przebywa ≥1,5 m w 4 s bez upadku, a przy `wz=±0,8` obraca się w zadaną stronę.

In [ ]:
RUN = "moj_wojtek"
kurs.trenuj(RUN, kroki=100_000_000, envs=4096)
moja = load_policy_runtime(kurs.eksportuj(RUN))
print("polityka:", moja.meta["run_name"], "| obserwacja:", moja.meta["obs_layout"])

In [ ]:
import ipywidgets as w
from IPython.display import display

vx = w.FloatSlider(0.5, min=-0.8, max=1.2, step=0.1, description="vx [m/s]")
vy = w.FloatSlider(0.0, min=-0.5, max=0.5, step=0.1, description="vy [m/s]")
wz = w.FloatSlider(0.0, min=-1.0, max=1.0, step=0.1, description="wz [rad/s]")
sek = w.IntSlider(4, min=2, max=10, description="sekundy")
przycisk, out = w.Button(description="Jedź"), w.Output()

def jedz(_):
    with out:
        out.clear_output()
        p = kurs.przebieg(moja, komenda=(vx.value, vy.value, wz.value), sekundy=sek.value, widok=widok)
        print(kurs.tabela({RUN: p}))
        show(p["klatki"])

przycisk.on_click(jedz)
display(w.VBox([vx, vy, wz, sek, przycisk, out]))

## Krok 8 — Finał: Twoja polityka kontra ta z robota

Polityki, które przeszły do użytku, leżą w repozytoriach Hugging Face `<organizacja>/<nazwa>` (checkpoint, konfiguracja, testy, wideo i para `policy.npz` + `policy_meta.json`). Robot ładuje taką politykę po nazwie i tym samym kodem, którego używa tu `kurs.przebieg`. Repozytoria są prywatne: w panelu Secrets Colaba (ikona klucza) dodaj `HF_ORGANIZATION` i `HF_TOKEN` i włącz dostęp dla tego notebooka.

Domyślne odniesienie to polityka przypięta na robocie (`wojtek-quiet-locomotion`). Oba przebiegi dostają ten sam scenariusz: stój, idź 0,5 m/s, obróć się 0,7 rad/s, idź, stój. Porównuj: przebytą drogę, błąd prędkości i obrotu, drgania, upadki, moment. Polityka z robota ma za sobą 2 mld kroków, randomizację masy, tarcia i opóźnień oraz testy na sprzęcie; różnica pokazuje, ile z tego dostaje się w godzinę na T4.

In [ ]:
try:
    from google.colab import userdata
    for k in ("HF_ORGANIZATION", "HF_TOKEN"):
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:
    pass
sys.path.insert(0, str(paths.WOJTEK_POLICY_PKG))
from wojtek_policy.policy_source import default_policy, resolve_policy

ORG = os.environ.get("HF_ORGANIZATION", "")
KEEPERS = ["wojtek-quiet-locomotion", "wojtek-stiff-height-locomotion", "wojtek-stiff-locomotion",
           "wojtek-stiff-locomotion-v2", "wojtek-stiff-kp80-locomotion", "wojtek-springy-locomotion-v2"]
if ORG:
    try:
        from huggingface_hub import HfApi
        KEEPERS = sorted(m.id.split("/", 1)[1] for m in HfApi().list_models(author=ORG)
                         if m.id.split("/", 1)[1].startswith("wojtek-")) or KEEPERS
    except Exception as e:
        print("lista z Hugging Face niedostępna:", type(e).__name__)
print("przypięta na robocie:", default_policy() or "(brak HF_ORGANIZATION w Secrets)")
wybor = w.Dropdown(options=KEEPERS, value=KEEPERS[0], description="polityka")
display(wybor)

In [ ]:
pin = default_policy()
ref = pin if pin and pin.split("/")[1].split("@")[0] == wybor.value else (f"{ORG}/{wybor.value}" if ORG else "")
if not ref:
    raise SystemExit("dodaj HF_ORGANIZATION i HF_TOKEN w Secrets, potem uruchom ponownie")
robot = load_policy_runtime(ref)
print("z robota:", resolve_policy(ref).source, "| serwo", robot.meta["pd"])

def scenariusz(i):                       # stój 2 s, przód 4 s, obrót 4 s, przód 4 s, stój 2 s
    if i < 100: return (0.0, 0.0, 0.0)
    if i < 300: return (0.5, 0.0, 0.0)
    if i < 500: return (0.0, 0.0, 0.7)
    if i < 700: return (0.5, 0.0, 0.0)
    return (0.0, 0.0, 0.0)

przebiegi = {"moja": kurs.przebieg(moja, scenariusz, sekundy=16, widok=widok),
             "robot": kurs.przebieg(robot, scenariusz, sekundy=16, widok=widok)}
print(kurs.tabela(przebiegi))
show(kurs.obok_siebie(przebiegi))        # po lewej Twoja, po prawej z robota

## Co dalej

- Więcej kroków, inne wagi, własne komendy: wszystko przez `kurs.trenuj(..., dodatkowe=...)`. Pełna lista nadpisań: `training/docs/configuration.md`.
- Ocena bez oglądania: `python -m wojtek_rl.report --run runs/<nazwa>` (bateria testów) i `python -m wojtek_rl.courses --run runs/<nazwa>` (tor przeszkód). Wnioski z poprzednich iteracji: `skills/brax-locomotion-training/references/wojtek-training-lessons.md`.
- Na robota polityka trafia przez `./ros/deploy.sh --policy <organizacja/nazwa@commit>` i ręczne uzbrojenie. To krok człowieka, nigdy notebooka.